# Jamsil Station 2024 Weather Collection

Collect hourly 2024 weather data for Jamsil Station from Open-Meteo, standardize the weather labels, and save the result to `Data/jamsil_2024.csv`.

In [ ]:
from __future__ import annotations

import json
import ssl
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

JAMSIL_STATION = {
    "name": "Jamsil Station",
    "latitude": 37.51333,
    "longitude": 127.10028,
}
API_URL = "https://archive-api.open-meteo.com/v1/archive"
TIMEZONE = "Asia/Seoul"
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
START_HOUR = 7
END_HOUR = 22
HOURLY_FIELDS = "weather_code,temperature_2m,wind_speed_10m,precipitation,rain,snowfall"
CAFILE_CANDIDATES = [
    Path("/etc/ssl/cert.pem"),
    Path("/private/etc/ssl/cert.pem"),
]
CLEAR_CODES = {0, 1}
SNOW_CODES = {71, 73, 75, 77, 85, 86}
RAINFALL_CODES = {51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99}


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "Note").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
OUTPUT_PATH = PROJECT_ROOT / "Data" / "jamsil_2024.csv"
EXPECTED_TIMELINE = pd.date_range(
    f"{START_DATE} 00:00:00",
    f"{END_DATE} 23:00:00",
    freq="h",
)
EXPECTED_TIMELINE = EXPECTED_TIMELINE[
    (EXPECTED_TIMELINE.hour >= START_HOUR) & (EXPECTED_TIMELINE.hour <= END_HOUR)
]


def build_ssl_context() -> ssl.SSLContext:
    for cafile in CAFILE_CANDIDATES:
        if cafile.exists():
            return ssl.create_default_context(cafile=str(cafile))
    return ssl.create_default_context()


def fetch_json(base_url: str, params: dict[str, object], retries: int = 3, timeout: int = 60) -> dict:
    url = f"{base_url}?{urlencode(params)}"
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            with urlopen(url, timeout=timeout, context=build_ssl_context()) as response:
                return json.load(response)
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt == retries:
                break
            print(f"Request failed ({attempt}/{retries}) for {base_url}: {exc}. Retrying...")
            time.sleep(2)

    raise RuntimeError(f"Failed to fetch Open-Meteo data after {retries} attempts.") from last_error


def coerce_numeric_series(values: list[object] | None, size: int) -> pd.Series:
    if values is None:
        return pd.Series([pd.NA] * size, dtype="Float64")

    series = pd.to_numeric(pd.Series(values), errors="coerce")
    if len(series) < size:
        series = series.reindex(range(size))
    else:
        series = series.iloc[:size]

    return series.reset_index(drop=True).astype("Float64")


def is_in_code_set(value: object, codes: set[int]) -> bool:
    try:
        return int(value) in codes
    except (TypeError, ValueError):
        return False


def numeric_value(value: object) -> float:
    if pd.isna(value):
        return 0.0
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def classify_weather(weather_code: object, precipitation: object, rain: object, snowfall: object) -> str:
    if numeric_value(snowfall) > 0 or is_in_code_set(weather_code, SNOW_CODES):
        return "snowy"
    if (
        numeric_value(rain) > 0
        or numeric_value(precipitation) > 0
        or is_in_code_set(weather_code, RAINFALL_CODES)
    ):
        return "rainfall"
    if is_in_code_set(weather_code, CLEAR_CODES):
        return "clear"
    return "cloudy"


def fetch_jamsil_weather() -> pd.DataFrame:
    params = {
        "latitude": JAMSIL_STATION["latitude"],
        "longitude": JAMSIL_STATION["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": HOURLY_FIELDS,
        "timezone": TIMEZONE,
    }
    payload = fetch_json(API_URL, params=params)

    hourly = payload.get("hourly")
    if not isinstance(hourly, dict) or not hourly.get("time"):
        raise ValueError("Open-Meteo response did not include hourly weather data.")

    frame = pd.DataFrame(
        {"datetime": pd.to_datetime(pd.Series(hourly["time"]), errors="coerce")}
    )
    size = len(frame)
    frame["weather_code"] = coerce_numeric_series(hourly.get("weather_code"), size)
    frame["temperature_c"] = coerce_numeric_series(hourly.get("temperature_2m"), size)
    frame["wind_speed"] = coerce_numeric_series(hourly.get("wind_speed_10m"), size)
    frame["precipitation"] = coerce_numeric_series(hourly.get("precipitation"), size)
    frame["rain"] = coerce_numeric_series(hourly.get("rain"), size)
    frame["snowfall"] = coerce_numeric_series(hourly.get("snowfall"), size)

    frame = (
        frame.dropna(subset=["datetime"])
        .drop_duplicates(subset=["datetime"], keep="last")
        .sort_values("datetime")
        .reset_index(drop=True)
    )

    expected = pd.DataFrame({"datetime": EXPECTED_TIMELINE})
    frame = expected.merge(frame, on="datetime", how="left")

    frame["weather"] = frame.apply(
        lambda row: classify_weather(
            row["weather_code"],
            row["precipitation"],
            row["rain"],
            row["snowfall"],
        ),
        axis=1,
    )
    frame["date"] = frame["datetime"].dt.strftime("%Y-%m-%d")
    frame["hour"] = frame["datetime"].dt.strftime("%H")
    frame["datetime"] = frame["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    frame["temperature_c"] = frame["temperature_c"].round(2)
    frame["wind_speed"] = frame["wind_speed"].round(2)

    return frame[["date", "hour", "datetime", "weather", "temperature_c", "wind_speed"]]


if OUTPUT_PATH.exists():
    print(f"Overwriting existing file: {OUTPUT_PATH}")

weather_df = fetch_jamsil_weather()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
weather_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")
print(
    f"Location: {JAMSIL_STATION['name']} ({JAMSIL_STATION['latitude']}, {JAMSIL_STATION['longitude']})"
)
print()
print(weather_df.head())
print()
print(f"Final row count: {len(weather_df)}")
print(f"Expected row count: {len(EXPECTED_TIMELINE)}")

missing_summary = weather_df[["temperature_c", "wind_speed"]].isna().sum()
if missing_summary.any():
    print()
    print("Missing value summary:")
    print(missing_summary[missing_summary > 0])
